In [118]:
import urllib.request
import os
import re
import glob
from bs4 import BeautifulSoup
import hashlib

Testing RSS

In [1]:
url = "https://www.goodreads.com/review/list_rss/56197581?key=cFzeoclEvdAOefh4RT7IodHnm3fI7hhI88CLXTDWxZz6AwrW&shelf=business"

# Fetch and print the raw RSS feed
response = urllib.request.urlopen(url)
feed = response.read().decode("utf-8")

# Save the raw RSS data to a file for inspection
with open("rss_sample.xml", "w", encoding="utf-8") as f:
    f.write(feed)

print("RSS data saved to rss_sample.xml for inspection.")

RSS data saved to rss_sample.xml for inspection.


Deleting empty notes

In [184]:
shelves = {
    "Biographies": "/Users/SFL/notes/content/Other Genres/Biographies",
    "Business Thinking": "/Users/SFL/notes/content/Self-help/Business Thinking",
    "Fiction": "/Users/SFL/notes/content/Other Genres/Fiction",
    "Humanities": "/Users/SFL/notes/content/Other Genres/Humanities",
    "Personal Development": "/Users/SFL/notes/content/Self-help/Personal Development",
    "Natural & Social Sciences": "/Users/SFL/notes/content/Natural & Social Sciences"
}

exclude_files = {
    "Natural & Social Sciences.md",
    "Personal Development.md",
    "Business Thinking.md",
    "Biographies.md",
    "Fiction.md",
    "Humanities.md",
    "index.md"
}

def is_empty_or_generated_only(file_path):
    # return True
    """Checks if a file contains only auto-generated content (subtitles) and no actual written content."""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read().strip()
            content = re.sub(r"^---.*?---", "", content, flags=re.DOTALL).strip()
            subtitle_pattern = re.compile(r"^## — .+", re.MULTILINE)
            content_without_subtitles = subtitle_pattern.sub("", content).strip()

            return len(content_without_subtitles) == 0

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return False  # Default to not deleting if there's an error

def clear_folder(folder_path):
    """Deletes only notes that are empty or contain only generated subtitles."""
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename in exclude_files:
                continue
            
            file_path = os.path.join(folder_path, filename)

            try:
                if os.path.isfile(file_path) and filename.endswith(".md"):
                    if is_empty_or_generated_only(file_path):
                        os.remove(file_path)
                    else:
                        print(f"Kept: {os.path.splitext(filename)[0]}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
    else:
        print(f"Folder does not exist: {folder_path}")

for category, folder_path in shelves.items():
    clear_folder(folder_path)

print("Cleanup completed!")

Cleanup completed!


Compile file link list

In [191]:
def update_shelf_notes(shelves):
    for shelf_name, folder_path in shelves.items():
        main_note_path = os.path.join(folder_path, "index.md")  # Ensure matching case

        if not os.path.isfile(main_note_path):
            
            # Create a new file named 'index.md' at the same directory
            index_note_path = os.path.join(os.path.dirname(main_note_path), "index.md")
            
            with open(index_note_path, "w", encoding="utf-8") as f:
                f.write(f"""---
title: {shelf_name.capitalize()}
---

""")  # Initial content
            print(f"Created new file: {index_note_path}")


        # Collect titles of all other .md files in the folder
        book_titles = [
            os.path.splitext(filename)[0]  # Remove .md extension
            for filename in os.listdir(folder_path)
            if filename.endswith(".md") and filename != "index.md"  # Exclude the main note
        ]

        # Format book titles as Obsidian links (sorted A-Z)
        book_links = "\n".join([f"[[{title}]]" for title in sorted(book_titles)])

        # Update the main note
        try:
            with open(main_note_path, "r", encoding="utf-8") as file:
                content = file.read()

            # Find the marker string and truncate everything after it
            marker = "**Full List of Books in this Category A-Z**"
            if marker in content:
                content = content.split(marker)[0] + marker + "\n" + book_links  # No extra newline
            else:
                content += f"\n\n{marker}\n" + book_links  # Ensure marker exists, no extra line

            # Write the updated content back to the main note
            with open(main_note_path, "w", encoding="utf-8") as file:
                file.write(content)

            print(f"Updated note: {main_note_path}")
        except Exception as e:
            print(f"Error updating {main_note_path}: {e}")

# Define the corrected shelf directories
shelves = {
    "Biographies": "/Users/SFL/notes/content/Other Genres/Biographies",
    "Business Thinking": "/Users/SFL/notes/content/Self-help/Business Thinking",
    "Fiction": "/Users/SFL/notes/content/Other Genres/Fiction",
    "Humanities": "/Users/SFL/notes/content/Other Genres/Humanities",
    "Personal Development": "/Users/SFL/notes/content/Self-help/Personal Development",
    "Natural & Social Sciences": "/Users/SFL/notes/content/Natural & Social Sciences"
}

# Run the function
update_shelf_notes(shelves)

Updated note: /Users/SFL/notes/content/Other Genres/Biographies/index.md
Created new file: /Users/SFL/notes/content/Self-help/Business Thinking/index.md
Updated note: /Users/SFL/notes/content/Self-help/Business Thinking/index.md
Created new file: /Users/SFL/notes/content/Other Genres/Fiction/index.md
Updated note: /Users/SFL/notes/content/Other Genres/Fiction/index.md
Created new file: /Users/SFL/notes/content/Other Genres/Humanities/index.md
Updated note: /Users/SFL/notes/content/Other Genres/Humanities/index.md
Created new file: /Users/SFL/notes/content/Self-help/Personal Development/index.md
Updated note: /Users/SFL/notes/content/Self-help/Personal Development/index.md
Updated note: /Users/SFL/notes/content/Natural & Social Sciences/index.md


Importing highlights

In [128]:
OBSIDIAN_DIR = "/Users/SFL/notes/content"

def generate_hash(text):
    """Generates a short hash for a given text."""
    return hashlib.md5(text.encode()).hexdigest()[:6]

def extract_book_title_and_quotes(html_path):
    """Extracts the book title and formatted highlighted quotes from a Kindle HTML file."""
    with open(html_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    book_title = soup.find("div", class_="bookTitle")
    book_title = book_title.get_text(strip=True) if book_title else None

    highlights = []
    last_highlight_index = None  # Tracks the last added highlight for nesting notes

    # Iterate through notes and highlights
    note_blocks = soup.find_all("div", class_=["noteHeading", "noteText"])
    
    for i in range(len(note_blocks) - 1):
        if "noteHeading" in note_blocks[i]["class"]:
            meta_info = note_blocks[i].get_text(strip=True)
            text = note_blocks[i + 1].get_text(strip=True) if "noteText" in note_blocks[i + 1]["class"] else ""

            if meta_info.startswith("Highlight"):  # Process as a highlight
                page_match = re.search(r"Page (\d+)", meta_info)
                page_info = f" – Page {page_match.group(1)}" if page_match else ""

                formatted_quote = f"> {text}{page_info}"
                highlights.append(formatted_quote)
                last_highlight_index = len(highlights) - 1  # Track last highlight

            elif meta_info.startswith("Note") and last_highlight_index is not None:  # Process as a nested note
                formatted_note = f"> > =={text}=="
                highlights[last_highlight_index] += f"\n{formatted_note}"  # Append to previous highlight

    return book_title, highlights
def find_matching_note(book_title):
    """Finds the best-matching note based on the beginning of the HTML title."""
    b = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", book_title)
    note_files = glob.glob(f"{OBSIDIAN_DIR}/**/*.md", recursive=True)
    note_titles = {os.path.splitext(os.path.basename(f))[0]: f for f in note_files}

    for note_title in note_titles:
        n = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", note_title)
        if b.lower().startswith(n.lower()):
            return note_titles[note_title]

    return None

def parse_existing_highlights(note_path):
    """Parses existing highlights and creates hash mappings for notes & identifiers."""
    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    in_select_quotes = False
    preserved_hashes = set()
    highlight_extra_map = {}

    lines = content.splitlines()
    lines = [s for s in lines if s.strip()]
    
    for i, line in enumerate(lines):
        if "## Select Quotes" in line:
            in_select_quotes = True
            continue

        if in_select_quotes:

            if line.startswith("> ") and not line.startswith("> >"):
                hash = generate_hash(line)

                has_note = bool(i + 1 < len(lines) and lines[i + 1].strip().startswith("> >"))
                has_id = bool(i + 1 < len(lines) and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 1].strip()))
                has_both = bool(i + 2 < len(lines) and lines[i + 1].strip().startswith("> >") and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 2].strip()))

                if has_both:
                    preserved_hashes.add(hash)
                    highlight_extra_map[hash] = lines[i + 1] + '\n' + lines[i + 2]
                elif has_note or has_id:
                    preserved_hashes.add(hash)
                    highlight_extra_map[hash] = lines[i + 1]
                
    return preserved_hashes, highlight_extra_map

def update_obsidian_note(note_path, highlights):
    """Updates the given Obsidian note with extracted highlights while preserving manually added notes and identifiers."""
    preserved_hashes, highlight_extra_map = parse_existing_highlights(note_path)

    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    if "## Select Quotes" in content:
        content = re.sub(r"## Select Quotes\n.*", "## Select Quotes", content, flags=re.DOTALL).strip()
    else:
        content += "\n## Select Quotes\n"

    new_content = []

    for i, line in enumerate(highlights):

        if line.startswith("> ") and not line.startswith("> >"): # Is a highlight
            new_content.append('\n' + line)
            highlight_hash = generate_hash(line)
            if highlight_hash in preserved_hashes: # in hash → attach it and its followings
                new_content.append(highlight_extra_map[highlight_hash]) 
        elif line.startswith("> >"):
            previous_hash = generate_hash(highlights[i - 1])
            if previous_hash in preserved_hashes and '> >' in highlight_extra_map[previous_hash]: # The previous highlight has a local note already
                continue
            else: # The previous highlight has no local note
                new_content.append(line)

    new_content = "\n".join(new_content) + "\n"

    with open(note_path, "w", encoding="utf-8") as file:
        file.write(content + '\n' + new_content)


def process_kindle_highlights(html_path):
    """Extracts Kindle highlights and updates the corresponding Obsidian note."""
    book_title, highlights = extract_book_title_and_quotes(html_path)

    if not book_title:
        print(f"Could not extract book title from {html_path}")
        return

    note_path = find_matching_note(book_title)
    
    if not note_path:
        print(f"No matching Obsidian note found for: {book_title}")
        return

    update_obsidian_note(note_path, highlights)
    print(f"Updated: {book_title}")

Process all highlights

In [ ]:
HIGHLIGHTS_DIR = "/Users/SFL/notes/private/highlights"

def process_all_highlights():
    # Iterate over all files in the directory
    for file_name in os.listdir(HIGHLIGHTS_DIR):
        file_path = os.path.join(HIGHLIGHTS_DIR, file_name)
        
        # Process only files (skip directories or invalid files)
        if os.path.isfile(file_path) and file_name.endswith(".html"):
            process_kindle_highlights(file_path)

# Run the function
process_all_highlights()

In [196]:
def extract_book_title_and_quotes(html_path):
    """Extracts the book title and formatted highlighted quotes from a Kindle HTML file."""
    with open(html_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    book_title = soup.find("div", class_="bookTitle")
    book_title = book_title.get_text(strip=True) if book_title else None

    return book_title

def find_matching_note(book_title):
    """Finds the best-matching note based on the beginning of the HTML title."""
    b = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", book_title)
    note_files = glob.glob(f"{OBSIDIAN_DIR}/**/*.md", recursive=True)
    note_titles = {os.path.splitext(os.path.basename(f))[0]: f for f in note_files}

    for note_title in note_titles:
        n = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", note_title)
        if b.lower().startswith(n.lower()):
            return True, (note_title, book_title)
            # return note_titles[note_title]  # Return the matched note path

    # return None
    return False, (book_title, note_title)

In [197]:
HIGHLIGHTS_DIR = "/Users/SFL/notes/private/highlights"
def process_all_highlights():

    results = {'success': [], 'failed': []}
    for file_name in os.listdir(HIGHLIGHTS_DIR):
        file_path = os.path.join(HIGHLIGHTS_DIR, file_name)
        
        if os.path.isfile(file_path) and file_name.endswith(".html"):
            book_title = extract_book_title_and_quotes(file_path)
            result = find_matching_note(book_title)
            if result[0]:
                results['success'].append(result[1])
            else:
                results['failed'].append(result[1][0])
    return results

results = process_all_highlights()

In [198]:
results['success']

[('Parasite Rex', 'Parasite Rex - Carl Zimmer'),
 ('Memory',
  'Memory: A Very Short Introduction (Very Short Introductions Book 194)'),
 ('Rich Dad, Poor Dad', 'Rich Dad Poor Dad'),
 ('Alcohol Explained', 'Alcohol Explained'),
 ('Know My Name', 'Know My Name_ A Memoir'),
 ('The Better Angels of Our Nature',
  'The Better Angels of Our Nature: Why Violence Has Declined'),
 ('Coming Up for Air', 'Coming Up for Air - Tom Daley'),
 ('Guns, Germs, and Steel',
  'Guns, Germs, and Steel The Fates of Human Societies, 20th anniversary edition -\xa0Jared Diamond (Z-Library)'),
 ('Chaos', 'Chaos: Making a New Science'),
 ('Counselling for Toads', 'Counselling for Toads'),
 ("She Has Her Mother's Laugh",
  "She Has Her Mother's Laugh_ The Powers, Pe - Carl Zimmer"),
 ('Crying in H Mart', 'Crying in H Mart_ A Memoir'),
 ('Educated', 'Educated - Tara Westover'),
 ('Walden', 'Walden'),
 ('Bad Feminist', 'Bad Feminist_ Essays'),
 ("Why Zebras Don't Get Ulcers",
  "Why Zebras Don't Get Ulcers _ The Ac

In [199]:
results['failed']

['Gu Du Liu Jiang',
 'Astrophysics: A Very Short Introduction (Very Short Introductions)',
 'Sheng Huo Shi Jiang']